# LangGraph 运行时¶
Pregel实现了 LangGraph 的运行时，管理 LangGraph 应用程序的执行

编译StateGraph或创建入口点会生成一个可以通过输入调用的Pregel实例。

本指南从高层次解释了运行时，并提供了使用 Pregel 直接实现应用程序的说明。

## 概述¶
在 LangGraph 中，Pregel 将Actor和Channel组合成一个应用程序。Actor从 Channel 读取数据，并将数据写入 Channel。Pregel 将应用程序的执行组织为多个步骤，遵循Pregel 算法/批量同步并行模型。

每个步骤包含三个阶段：

- 计划：确定在此步骤中执行哪些Actor。例如，在第一步中，选择订阅特殊输入频道的Actor；在后续步骤中，选择订阅上一步中更新的频道的Actor 。
- 执行：并行执行所有选定的Actor，直到所有 Actor 完成、一个 Actor 失败或超时。在此阶段，通道更新对 Actor 不可见，直到下一步执行。
- 更新：使用参与者在此步骤中写入的值更新通道。
重复此操作，直到没有选择任何参与者执行，或者达到最大步骤数

## Actor¶
`Actor`是一个 `PregelNode`。它订阅通道，从中读取数据，并向通道写入数据。它可以被视为 Pregel 算法中的一个角色。`PregelNodes` 实现了 LangChain 的 Runnable 接口。

## 频道¶
通道用于在参与者（PregelNodes）之间进行通信。每个通道都有一个值类型、一个更新类型和一个更新函数——该函数接受一系列更新操作并修改存储的值。通道可用于将数据从一个链发送到另一个链，或者在未来的步骤中将数据从一个链发送到自身。LangGraph 提供了许多内置通道：

- LastValue：默认通道，存储发送到通道的最后一个值，可用于输入和输出值，或用于将数据从一个步骤发送到下一个步骤。
- 主题 (Topic ) ：可配置的 PubSub 主题，用于在参与者之间发送多个值，或用于累积输出。可以配置为删除重复值或在多个步骤中累积值。
- BinaryOperatorAggregate：存储一个持久值，通过对当前值应用二进制运算符和发送到通道的每次更新来更新，对于计算多个步骤的聚合很有用;例如，total = BinaryOperatorAggregate（int， operator.add）

## 示例¶
虽然大多数用户将通过StateGraph API 或入口点装饰器与 Pregel 交互，但也可以直接与 Pregel 交互。

下面是一些不同的例子，让您了解 Pregel API。

In [ ]:
from langgraph.channels import EphemeralValue
from langgraph.pregel import Pregel, NodeBuilder

node1 = (
    NodeBuilder().subscribe_only("a")
    .do(lambda x: x + x)
    .write_to("b")
)

app = Pregel(
    nodes={"node1": node1},
    channels={
        "a": EphemeralValue(str),
        "b": EphemeralValue(str),
    },
    input_channels=["a"],
    output_channels=["b"],
)

app.invoke({"a": "foo"})

In [ ]:
from langgraph.channels import LastValue, EphemeralValue
from langgraph.pregel import Pregel, NodeBuilder

node1 = (
    NodeBuilder().subscribe_only("a")
    .do(lambda x: x + x)
    .write_to("b")
)

node2 = (
    NodeBuilder().subscribe_only("b")
    .do(lambda x: x + x)
    .write_to("c")
)


app = Pregel(
    nodes={"node1": node1, "node2": node2},
    channels={
        "a": EphemeralValue(str),
        "b": LastValue(str),
        "c": EphemeralValue(str),
    },
    input_channels=["a"],
    output_channels=["b", "c"],
)

app.invoke({"a": "foo"})

In [ ]:
from langgraph.channels import EphemeralValue, Topic
from langgraph.pregel import Pregel, NodeBuilder

node1 = (
    NodeBuilder().subscribe_only("a")
    .do(lambda x: x + x)
    .write_to("b", "c")
)

node2 = (
    NodeBuilder().subscribe_to("b")
    .do(lambda x: x["b"] + x["b"])
    .write_to("c")
)

app = Pregel(
    nodes={"node1": node1, "node2": node2},
    channels={
        "a": EphemeralValue(str),
        "b": EphemeralValue(str),
        "c": Topic(str, accumulate=True),
    },
    input_channels=["a"],
    output_channels=["c"],
)

app.invoke({"a": "foo"})

## 高级 API¶
LangGraph 为创建 Pregel 应用程序提供了两个高级 API：StateGraph（Graph API）和Functional API。

StateGraph （Graph API）是一种更高级别的抽象，可以简化 Pregel 应用程序的创建。它允许您定义由节点和边组成的图。编译该图时，StateGraph API 会自动为您创建 Pregel 应用程序。

In [ ]:
from typing import TypedDict, Optional

from langgraph.constants import START
from langgraph.graph import StateGraph

class Essay(TypedDict):
    topic: str
    content: Optional[str]
    score: Optional[float]

def write_essay(essay: Essay):
    return {
        "content": f"Essay about {essay['topic']}",
    }

def score_essay(essay: Essay):
    return {
        "score": 10
    }

builder = StateGraph(Essay)
builder.add_node(write_essay)
builder.add_node(score_essay)
builder.add_edge(START, "write_essay")

# Compile the graph.
# This will return a Pregel instance.
graph = builder.compile()

编译后的 Pregel 实例将与节点和通道列表关联。您可以通过打印来检查这些节点和通道。



In [ ]:
print(graph.nodes)



```shell
{'__start__': <langgraph.pregel.read.PregelNode at 0x7d05e3ba1810>,
 'write_essay': <langgraph.pregel.read.PregelNode at 0x7d05e3ba14d0>,
 'score_essay': <langgraph.pregel.read.PregelNode at 0x7d05e3ba1710>}
```

In [1]:
print(graph.channels)
## 你应该看到类似这样的内容




NameError: name 'graph' is not defined

```shell
{'topic': <langgraph.channels.last_value.LastValue at 0x7d05e3294d80>,
 'content': <langgraph.channels.last_value.LastValue at 0x7d05e3295040>,
 'score': <langgraph.channels.last_value.LastValue at 0x7d05e3295980>,
 '__start__': <langgraph.channels.ephemeral_value.EphemeralValue at 0x7d05e3297e00>,
 'write_essay': <langgraph.channels.ephemeral_value.EphemeralValue at 0x7d05e32960c0>,
 'score_essay': <langgraph.channels.ephemeral_value.EphemeralValue at 0x7d05e2d8ab80>,
 'branch:__start__:__self__:write_essay': <langgraph.channels.ephemeral_value.EphemeralValue at 0x7d05e32941c0>,
 'branch:__start__:__self__:score_essay': <langgraph.channels.ephemeral_value.EphemeralValue at 0x7d05e2d88800>,
 'branch:write_essay:__self__:write_essay': <langgraph.channels.ephemeral_value.EphemeralValue at 0x7d05e3295ec0>,
 'branch:write_essay:__self__:score_essay': <langgraph.channels.ephemeral_value.EphemeralValue at 0x7d05e2d8ac00>,
 'branch:score_essay:__self__:write_essay': <langgraph.channels.ephemeral_value.EphemeralValue at 0x7d05e2d89700>,
 'branch:score_essay:__self__:score_essay': <langgraph.channels.ephemeral_value.EphemeralValue at 0x7d05e2d8b400>,
 'start:write_essay': <langgraph.channels.ephemeral_value.EphemeralValue at 0x7d05e2d8b280>}

 返回顶部
以前的
使用函数式 API

```